In [1]:
import polars as pl
import numpy as np
import os
import pandas as pd

DATA_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\dataset\sales_pers.item_chunk_0.parquet"

df = pl.read_parquet(DATA_PATH)
print(f"Số dòng: {df.height:,}, Số cột: {df.width}")


Số dòng: 27,332, Số cột: 34


In [2]:
print("\nCác cột ban đầu:")
print(df.columns)


Các cột ban đầu:
['p_id', 'item_id', 'price', 'category_l1_id', 'category_l1', 'category_l2_id', 'category_l2', 'category_l3_id', 'category_l3', 'category_id', 'category', 'description', 'brand', 'manufacturer', 'creation_timestamp', 'is_deleted', 'created_date', 'updated_date', 'sync_status_id', 'last_sync_date', 'sync_error_message', 'image_url', 'gender_target', 'age_group', 'item_type', 'gp', 'weight', 'color', 'size', 'origin', 'volume', 'material', 'sale_status', 'description_new']


# Task 1: Loại bỏ các cột mà nhóm nghĩ là không cần thiết.

In [3]:
# --- Danh sách cột cần loại bỏ (theo phân tích EDA & tương quan) ---
cols_to_drop = [
    # --- Metadata hệ thống (không dùng cho mô hình) ---
    "p_id",
    "is_deleted",
    "sync_status_id",
    "sync_error_message",
    "image_url",
    "last_sync_date",
    "creation_timestamp",
    "updated_date",
    "created_date",
    "sale_status",

    # --- Cột numeric hầu như vô nghĩa / chỉ có 1 giá trị ---
    "gp",
    "weight",
    "volume",

    # --- Các cột chất lượng kém / missing cực cao / không dùng ---
    "material",
    "origin",
    "manufacturer",
    "color",
    "size",

    # --- ID phân loại (trùng với tên category) ---
    "category_l1_id",
    "category_l2_id",
    "category_l3_id",
    "category_id"
]



# --- Loại bỏ các cột không cần thiết ---
df_cleaned = df.drop(cols_to_drop)

print(f"\nĐã loại bỏ {len(cols_to_drop)} cột không cần thiết.")
print(f"Số cột còn lại: {df_cleaned.width}")
print("\nDanh sách cột sau khi loại bỏ:")
print(df_cleaned.columns)



Đã loại bỏ 22 cột không cần thiết.
Số cột còn lại: 12

Danh sách cột sau khi loại bỏ:
['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'description', 'brand', 'gender_target', 'age_group', 'item_type', 'description_new']


# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý outlier

In [4]:
# df_cleaned: dataframe sau khi đã drop các cột không cần thiết
print("Số lượng ban đầu:", df_cleaned.height)

# --- LOẠI SẢN PHẨM CÓ PRICE ≤ 1 (Outlier) ---
df_no_outlier = df_cleaned.filter(pl.col("price") > 1)

print("Số lượng sau khi loại outlier (price ≤ 1):", df_no_outlier.height)
print("Số lượng bị loại:", df_cleaned.height - df_no_outlier.height)

# Kiểm tra lại xem còn outlier hay không
print("\nKiểm tra min price sau khi xử lý:")
print(df_no_outlier.select(pl.col("price").min()))

Số lượng ban đầu: 27332
Số lượng sau khi loại outlier (price ≤ 1): 27323
Số lượng bị loại: 9

Kiểm tra min price sau khi xử lý:
shape: (1, 1)
┌───────────────┐
│ price         │
│ ---           │
│ decimal[38,4] │
╞═══════════════╡
│ 1000.0000     │
└───────────────┘


In [5]:
# Chuẩn hóa giá trị Unisex → Không xác định
df_no_outlier = df_no_outlier.with_columns(
    pl.col("gender_target").replace("Unisex", "Không xác định")
)

print("Đã chuyển toàn bộ Unisex thành 'Không xác định'.")
print(df_no_outlier["gender_target"].value_counts())


Đã chuyển toàn bộ Unisex thành 'Không xác định'.
shape: (4, 2)
┌────────────────┬───────┐
│ gender_target  ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ Sơ sinh        ┆ 1862  │
│ Bé Trai        ┆ 3318  │
│ Không xác định ┆ 18035 │
│ Bé Gái         ┆ 4108  │
└────────────────┴───────┘


In [6]:
print(f"Số dòng: uj {df_no_outlier.height:,}, Số cột: {df_no_outlier.width}")
df_no_outlier.head(5)

Số dòng: uj 27,323, Số cột: 12


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …"
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …"
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""


## Xử lý null

In [7]:
pl.Config.set_tbl_rows(50)             # số dòng tối đa hiển thị


# df_final = df_no_outlier
df_final = df_no_outlier

n = df_final.height

results = []

for col in df_final.columns:
    s = df_final[col]

    # NULL
    null_count = s.null_count()

    # UNKNOWN ("Không xác định")
    if s.dtype == pl.String:
        # Đếm trực tiếp bằng Python list, không lỗi
        unknown_count = s.to_list().count("Không xác định")
    else:
        unknown_count = 0

    results.append({
        "column": col,
        "dtype": s.dtype,
        "null_count": null_count,
        "null_ratio(%)": round(null_count / n * 100, 2),
        "unknown_count": unknown_count,
        "unknown_ratio(%)": round(unknown_count / n * 100, 2)
    })

df_missing_report = pl.DataFrame(results)

print(df_missing_report)


shape: (12, 6)
┌─────────────────┬─────────────────┬────────────┬───────────────┬───────────────┬─────────────────┐
│ column          ┆ dtype           ┆ null_count ┆ null_ratio(%) ┆ unknown_count ┆ unknown_ratio(% │
│ ---             ┆ ---             ┆ ---        ┆ ---           ┆ ---           ┆ )               │
│ str             ┆ object          ┆ i64        ┆ f64           ┆ i64           ┆ ---             │
│                 ┆                 ┆            ┆               ┆               ┆ f64             │
╞═════════════════╪═════════════════╪════════════╪═══════════════╪═══════════════╪═════════════════╡
│ item_id         ┆ String          ┆ 0          ┆ 0.0           ┆ 0             ┆ 0.0             │
│ price           ┆ Decimal(precisi ┆ 0          ┆ 0.0           ┆ 0             ┆ 0.0             │
│                 ┆ on=38, scale=4) ┆            ┆               ┆               ┆                 │
│ category_l1     ┆ String          ┆ 0          ┆ 0.0           ┆ 0        

### Xử lý gender_target

In [8]:

# Lọc các item thời trang nhưng gender_target = "Không xác định"
df_fashion_unknown = (
    df_no_outlier
        .filter(
            (pl.col("category_l1") == "Thời trang") &
            (pl.col("gender_target") == "Không xác định")
        )
)

print("Tổng số item thời trang có gender_target = 'Không xác định':", df_fashion_unknown.height)

# Lấy 20 dòng ngẫu nhiên
df_sample = df_fashion_unknown.sample(n=20, with_replacement=False)

pd.set_option('display.max_columns', None)
df_sample.head(10)

Tổng số item thời trang có gender_target = 'Không xác định': 7264


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""0012050000027""",29000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần lót bầu""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0849019140002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bodysuit""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0011050160060""",9000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Phụ kiện khác""","""Vớ""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6996000000133""",199000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bộ Modal""","""Bộ Modal lẻ""","""Không xác định""","""Animo""","""Không xác định""","""9M-12M""","""Bộ quần áo""","""Không xác định"""
"""3390000000025""",179000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bodysuit Modal""","""Bodysuit Modal lẻ""","""Không xác định""","""Animo""","""Không xác định""","""9M-12M""","""Bodysuit""","""Không xác định"""
"""0009010040186""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Áo sơ sinh""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Từ 1Y""","""Áo""","""Không xác định"""
"""6034000000326""",129000.0000,"""Thời trang""","""Thời trang bé gái""","""Bộ bé gái""","""Bộ bé gái Animo Easy""","""﻿Bộ bé gái ngắn Animo Easy TX0…","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩmTên sản phẩm:…"
"""0987017510003""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bodysuit""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3522104200001""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Áo sơ sinh""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null


In [9]:
# Lọc sản phẩm thời trang có category chứa "đầm" hoặc "váy" và gender_target = "Không xác định"
df_dam_vay_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Thời trang")
        &
        (pl.col("gender_target") == "Không xác định")
        &
        (
            pl.col("category").str.contains("đầm", literal=False)
            | pl.col("category").str.contains("váy", literal=False)
        )
    )
)

# In thống kê số dòng
print("Tổng sản phẩm thỏa điều kiện:", df_dam_vay_unknown.height)

# Lấy mẫu 20 dòng
df_sample = df_dam_vay_unknown.sample(n=20, seed=42)

# Hiển thị đầy đủ cột
pd.set_option('display.max_columns', None)
df_sample.head(10)


Tổng sản phẩm thỏa điều kiện: 192


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""6045000000002""",349000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""1092093850001""",189000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Áo bầu""","""Không xác định"""
"""1083022850002""",149000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Quần bầu""","""Không xác định"""
"""0886026770002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0887026770004""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0891019790002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0012190160065""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Laluna""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""1080033860001""",199000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""1080004860001""",189000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Áo bầu""","""Không xác định"""


In [10]:
pl.Config.set_tbl_cols(20)             # số cột tối đa hiển thị

polars.config.Config

In [11]:
# ============================================
#  TẠO CỘT gender_target_final
# ============================================

df_filled = df_no_outlier.with_columns(
    pl.col("gender_target").alias("gender_target_final")
)

# ============================================
#  1) FILL “SƠ SINH”
# ============================================

mask_sosinh = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("gender_target_final") == "Không xác định")
    & (
        pl.col("category_l2").str.contains("sơ sinh", literal=False)
        | pl.col("category").str.contains("sơ sinh", literal=False)
        | pl.col("category").str.contains("0-3m", literal=False)
        | pl.col("category").str.contains("3-6m", literal=False)
        | pl.col("category").str.contains("0-12m", literal=False)
        | pl.col("category").str.contains(r"\bnb\b", literal=False)   # NB
        | pl.col("category").str.contains("newborn", literal=False)
    )
)

df_filled = df_filled.with_columns(
    pl.when(mask_sosinh)
      .then(pl.lit("Sơ sinh"))
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# ============================================
#  2) FILL “BÉ GÁI”
# ============================================

mask_fashion_unknown = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("gender_target_final") == "Không xác định")
)

mask_not_maternity = (
    (~pl.col("category").str.contains("bầu", literal=False))
    & (~pl.col("category_l2").str.contains("bầu", literal=False))
    & (~pl.col("category_l3").str.contains("bầu", literal=False))
)

mask_not_ambiguous = (
    ~pl.col("category_l3").str.contains("Thời trang bé trai, bé gái cũ", literal=False)
)

mask_girl_keyword = (
    pl.col("category").str.contains("bé gái|đầm|váy|chân váy", literal=False)
    | pl.col("category_l2").str.contains("bé gái|đầm bé gái|bộ bé gái", literal=False)
    | pl.col("category_l3").str.contains("bé gái|đầm|váy", literal=False)
)

mask_fill_girl = (
    mask_fashion_unknown
    & mask_not_maternity
    & mask_not_ambiguous
    & mask_girl_keyword
)

df_filled = df_filled.with_columns(
    pl.when(mask_fill_girl)
      .then(pl.lit("Bé Gái"))
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# ============================================
#  3) FILL “BÉ TRAI”
# ============================================

mask_boy_keyword = (
    pl.col("category").str.contains("bé trai", literal=False)
    | pl.col("category_l2").str.contains("bé trai|bộ bé trai", literal=False)
    | pl.col("category_l3").str.contains("bé trai", literal=False)
)

mask_fill_boy = (
    mask_fashion_unknown
    & mask_not_ambiguous
    & mask_boy_keyword
)

df_filled = df_filled.with_columns(
    pl.when(mask_fill_boy)
      .then(pl.lit("Bé Trai"))
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# ============================================
# 4) KIỂM TRA KẾT QUẢ FILL
# ============================================

print("\nTổng số dòng được fill:",
      df_filled.filter(pl.col("gender_target") != pl.col("gender_target_final")).height
)


total_rows = df_filled.height

unknown_original = df_filled.filter(
    pl.col("gender_target") == "Không xác định"
).height

unknown_final = df_filled.filter(
    pl.col("gender_target_final") == "Không xác định"
).height

ratio_unknown_final = unknown_final / total_rows * 100

print("\n=== THỐNG KÊ 'Không xác định' ===")
print(f"1) gender_target ban đầu = 'Không xác định': {unknown_original:,}")
print(f"2) gender_target_final    = 'Không xác định': {unknown_final:,}")
print(f"3) Tỷ lệ 'Không xác định' sau fill: {ratio_unknown_final:.2f}% (trên {total_rows:,} dòng)")

# Xuất ra 5 dòng mẫu sau khi fill
print("\n=== 5 dòng mẫu đã được fill ===")
print(
    df_filled
    .filter(
        (pl.col("gender_target") == "Không xác định")
        & (pl.col("gender_target_final") != "Không xác định")
    )
    .head(5)
)



Tổng số dòng được fill: 3333

=== THỐNG KÊ 'Không xác định' ===
1) gender_target ban đầu = 'Không xác định': 18,035
2) gender_target_final    = 'Không xác định': 14,702
3) Tỷ lệ 'Không xác định' sau fill: 53.81% (trên 27,323 dòng)

=== 5 dòng mẫu đã được fill ===
shape: (5, 13)
┌─────┬─────┬─────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ ite ┆ pri ┆ cat ┆ catego ┆ categ ┆ categ ┆ descr ┆ brand ┆ gende ┆ age_g ┆ item_ ┆ descr ┆ gende │
│ m_i ┆ ce  ┆ ego ┆ ry_l2  ┆ ory_l ┆ ory   ┆ iptio ┆ ---   ┆ r_tar ┆ roup  ┆ type  ┆ iptio ┆ r_tar │
│ d   ┆ --- ┆ ry_ ┆ ---    ┆ 3     ┆ ---   ┆ n     ┆ str   ┆ get   ┆ ---   ┆ ---   ┆ n_new ┆ get_f │
│ --- ┆ dec ┆ l1  ┆ str    ┆ ---   ┆ str   ┆ ---   ┆       ┆ ---   ┆ str   ┆ str   ┆ ---   ┆ inal  │
│ str ┆ ima ┆ --- ┆        ┆ str   ┆       ┆ str   ┆       ┆ str   ┆       ┆       ┆ str   ┆ ---   │
│     ┆ l[3 ┆ str ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆ str   │
│     ┆ 8,4 ┆

In [12]:
# Giữ lại toàn bộ dữ liệu đã fill
df_final = df_filled.drop("gender_target")

print("Số cột sau khi drop gender_target:", df_final.width)
print(df_final.columns)


Số cột sau khi drop gender_target: 12
['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'description', 'brand', 'age_group', 'item_type', 'description_new', 'gender_target_final']


### Xử lý age_group

In [13]:
# Lọc các sản phẩm Thời trang nhưng age_group = "Không xác định"
df_fashion_age_unknown = (
    df_final
    .filter(
        (pl.col("category_l1") == "Thời trang") &
        (pl.col("age_group") == "Không xác định")
    )
)

# Kiểm tra số lượng
total = df_fashion_age_unknown.height
print("Tổng số sản phẩm Thời trang có age_group = 'Không xác định':", total)

# -----------------------------
# Thống kê description & description_new
# -----------------------------

count_desc_unknown = df_fashion_age_unknown.filter(pl.col("description") == "Không xác định").height
count_desc_new_unknown = df_fashion_age_unknown.filter(pl.col("description_new") == "Không xác định").height

print("\nThống kê tình trạng mô tả trong nhóm này:")
print(f"- description = 'Không xác định': {count_desc_unknown} / {total} ({count_desc_unknown/total*100:.2f}%)")
print(f"- description_new = 'Không xác định': {count_desc_new_unknown} / {total} ({count_desc_new_unknown/total*100:.2f}%)")

# -----------------------------
# Lấy ngẫu nhiên 20 dòng để quan sát
# -----------------------------
df_sample_age = df_fashion_age_unknown.sample(n=20, seed=42)

pd.set_option('display.max_columns', None)
df_sample_age.head(5)


Tổng số sản phẩm Thời trang có age_group = 'Không xác định': 6058

Thống kê tình trạng mô tả trong nhóm này:
- description = 'Không xác định': 5435 / 6058 (89.72%)
- description_new = 'Không xác định': 2593 / 6058 (42.80%)


item_id,price,category_l1,category_l2,category_l3,category,description,brand,age_group,item_type,description_new,gender_target_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""3523000000021""",99000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Áo""","""Áo sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""Sơ sinh"""
"""3320016830007""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé trai""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""Không xác định"""
"""0930002360001""",19000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Tã vải""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Tã vải""","""Không xác định""","""Sơ sinh"""
"""3533000000241""",119000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Quần""","""Quần sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""Sơ sinh"""
"""6034000000360""",129000.0000,"""Thời trang""","""Thời trang bé gái""","""Bộ bé gái""","""Bộ bé gái Animo Easy""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Bé Gái"""


In [14]:
total = df_final.height

age_counts = (
    df_final
    .group_by("age_group")            # <-- đúng cú pháp Polars
    .agg(pl.count().alias("count"))
    .with_columns(
        (pl.col("count") / total * 100).round(2).alias("ratio_percent")
    )
    .sort("count", descending=True)
)

print(age_counts)


shape: (112, 3)
┌─────────────────────────────┬───────┬───────────────┐
│ age_group                   ┆ count ┆ ratio_percent │
│ ---                         ┆ ---   ┆ ---           │
│ str                         ┆ u32   ┆ f64           │
╞═════════════════════════════╪═══════╪═══════════════╡
│ Không xác định              ┆ 15803 ┆ 57.84         │
│ Từ 3Y                       ┆ 944   ┆ 3.45          │
│ 9M-12M                      ┆ 844   ┆ 3.09          │
│ 6M-9M                       ┆ 672   ┆ 2.46          │
│ Từ 1Y                       ┆ 609   ┆ 2.23          │
│ Từ 2Y                       ┆ 577   ┆ 2.11          │
│ 3M-6M                       ┆ 545   ┆ 1.99          │
│ Từ 6M                       ┆ 508   ┆ 1.86          │
│ 0-12M                       ┆ 431   ┆ 1.58          │
│ Từ 0M                       ┆ 430   ┆ 1.57          │
│ 12M-18M                     ┆ 429   ┆ 1.57          │
│ Từ 4Y                       ┆ 425   ┆ 1.56          │
│ Từ 9M                       ┆ 

C:\Users\PC\AppData\Local\Temp\ipykernel_9664\1541734142.py:6: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("count"))


#### Lấy brand và age_group từ Như và Thành

# Task 3: Phân tích tương đồng và xác định xem các thuộc tính tương tự nhau. Từ đó loại bỏ đặc trưng thừa.

Theo quá trình EDA, chúng tôi nhận thấy rằng một số đặc trưng trong bảng `item_chunk` thể hiện mức độ tương đồng rất cao, phản ánh sự trùng lặp thông tin hoặc cấu trúc phân cấp của dữ liệu. Dựa trên ma trận Cramér’s V và phân tích ngữ nghĩa của các trường, các nhóm đặc trưng tương đồng được xác định như sau:


##### **(1) Nhóm đặc trưng phân loại sản phẩm (Category Hierarchy Group)**  
Bao gồm các biến:
- `category_l1`
- `category_l2`
- `category_l3`
- `category` (leaf category – cấp chi tiết nhất)

Kết quả tương quan cho thấy các đặc trưng này có mức độ tương đồng rất cao (Cramér’s V từ **0.94 đến 0.97**). Điều này phản ánh đúng cấu trúc phân cấp của taxonomy sản phẩm:

- `category_l1` mô tả ngành hàng cấp cao nhất (macro-level)
- `category_l2` mô tả nhóm sản phẩm chi tiết hơn
- `category_l3` và `category` mô tả đến mức SKU-group rất chi tiết

Do đó, các biến này chứa thông tin lặp lại ở nhiều cấp độ khác nhau, và cần lựa chọn cấp độ phù hợp để giữ lại nhằm tránh đa cộng tuyến và giảm độ phức tạp mô hình.

---

##### **(2) Nhóm đặc trưng mô tả sản phẩm (Description Group)**  
Bao gồm:
- `description` (mô tả cũ)
- `description_new` (mô tả mới, được chuẩn hóa hơn)

Hai đặc trưng này thể hiện tương quan cao (**Cramér’s V ≈ 0.55–0.74**).  
Ngoài ra, `description` chứa rất nhiều bản ghi "Không xác định" (>18,000 dòng), dẫn đến chất lượng thấp và trùng lặp nội dung với `description_new`.  
Vì vậy, cần xem xét giữ lại phiên bản mô tả có chất lượng cao hơn.

---

##### **(3) Các đặc trưng quan hệ chặt chẽ với taxonomy sản phẩm**  
Một số biến khác cũng thể hiện tương quan đáng kể với nhóm category, nhưng mang ý nghĩa nội dung khác và không phải là bản sao trực tiếp. Chẳng hạn:
- `brand` ↔ `category` (V ≈ 0.79)
- `item_type` ↔ `category` (V ≈ 0.77)
- `gender_target` và `age_group` cũng liên quan đến nhóm ngành nhưng vẫn mang thông tin hành vi riêng biệt

Những đặc trưng này **không bị loại bỏ**, nhưng được ghi nhận vì có mức độ tương quan cao, nhằm lưu ý khi xây dựng mô hình tránh đa cộng tuyến ở các thuật toán tuyến tính hoặc tree-based truyền thống.


#### Quyết định xử lý các nhóm đặc trưng tương đồng

##### **1. Loại bỏ các cột `category_l3` và `category`**

**Lý do:**

- Kết quả phân tích Cramér’s V cho thấy `category_l3` và `category` có mức tương đồng rất cao so với `category_l1` và `category_l2` (từ **0.92 đến 0.97**). Điều này phản ánh rằng chúng gần như chứa cùng một lượng thông tin nhưng ở cấp độ chi tiết hơn.
- Hai trường này có **số lượng giá trị phân biệt lớn** (hundreds đến hàng nghìn), dẫn đến hiện tượng **high-cardinality**, gây:
  - sparsity trong vector encoding,
  - nguy cơ overfitting,
  - tăng chi phí tính toán mà không mang lại thêm thông tin hành vi mua hàng.
- Cấp `category_l3/category` mô tả nhóm sản phẩm đến mức SKU-group, trong khi mô hình dự đoán sản phẩm kế tiếp đã có đặc trưng `item_id` để thể hiện mức chi tiết này.
- Các cấp phân loại chi tiết không mang ý nghĩa bổ sung cho hành vi mua như category_l1 và category_l2.

---

##### **2. Gộp thông tin `description` vào `description_new`, sau đó loại bỏ `description`**

**Lý do:**

- Hai trường mô tả sản phẩm (`description` và `description_new`) có mức tương quan cao (**Cramér’s V ≈ 0.55–0.74**), cho thấy lượng thông tin chồng lấp rất lớn.
- `description` có chất lượng dữ liệu thấp: hơn **18.000 dòng mang giá trị “Không xác định”**, khiến trường này không thể sử dụng trực tiếp.
- `description_new` là phiên bản được chuẩn hóa hơn, nhưng vẫn tồn tại khoảng **5.000 bản ghi null**, nghĩa là vẫn cần bổ sung thông tin.
- Để tận dụng tối đa dữ liệu mô tả, nhóm thực hiện:
  1. **Merge thông tin hợp lệ từ `description` vào `description_new` đối với các dòng bị thiếu.**
  2. Ưu tiên giá trị từ `description_new` nếu cả hai cùng tồn tại.
- Sau khi hợp nhất, trường `description` không còn cần thiết.

### Thực hiện

In [ ]:
# ===============================
# 1. Loại bỏ category_l3 và category
# ===============================

#cols_to_drop_cat = ["category_l3", "category"]

#df_final = df_final.drop(cols_to_drop_cat)

#print("Sau khi drop category_l3 và category:")
#print(f"Số cột còn lại: {df_final.width}")
#print(df_final.columns)

Sau khi drop category_l3 và category:
Số cột còn lại: 12
['item_id', 'price', 'category_l1', 'category_l2', 'description', 'brand', 'age_group', 'item_type', 'color', 'size', 'description_new', 'gender_target_final']


In [ ]:
# ===============================
# 1. Tạo description_final
# ===============================

df_final = df_final.with_columns(
    pl.when(
        (pl.col("description_new").is_not_null())
        & (pl.col("description_new") != pl.lit("Không xác định"))
    )
    .then(pl.col("description_new"))
    .otherwise(
        pl.when(pl.col("description_new").is_null())
        .then(pl.lit("Không xác định"))
        .otherwise(pl.col("description"))
    )
    .alias("description_final")
)

print("\nTạo description_final xong")
print(f"Số cột hiện tại: {df_final.width}")
print(df_final.columns)



Tạo description_final xong
Số cột hiện tại: 13
['item_id', 'price', 'category_l1', 'category_l2', 'description', 'brand', 'age_group', 'item_type', 'color', 'size', 'description_new', 'gender_target_final', 'description_final']


In [ ]:
# ===============================
# 2. Tính tỷ lệ 'Không xác định'
# ===============================

total_rows = df_final.height
count_unknown = df_final.filter(
    pl.col("description_final") == "Không xác định"
).height

ratio_unknown = count_unknown / total_rows * 100

print(
    f"\nTỷ lệ 'Không xác định' trong description_final: "
    f"{ratio_unknown:.2f}% ({count_unknown}/{total_rows})"
)


Tỷ lệ 'Không xác định' trong description_final: 56.49% (15436/27323)


In [ ]:
df_filled_from_description = df_final.filter(
    (
        pl.col("description_new").is_null() |
        (pl.col("description_new") == "Không xác định")
    )
    &
    (pl.col("description").is_not_null()) &
    (pl.col("description") != "Không xác định") &
    (pl.col("description_final") == pl.col("description"))
).select([
    "item_id",
    "description_new",
    "description",
    "description_final"
])

print("\nCác dòng đã được fill mô tả từ description:")
print(df_filled_from_description.head(20))
print(f"Tổng số dòng fill được: {df_filled_from_description.height}")



Các dòng đã được fill mô tả từ description:
shape: (20, 4)
┌───────────────┬─────────────────┬────────────────────────────────┬───────────────────────────────┐
│ item_id       ┆ description_new ┆ description                    ┆ description_final             │
│ ---           ┆ ---             ┆ ---                            ┆ ---                           │
│ str           ┆ str             ┆ str                            ┆ str                           │
╞═══════════════╪═════════════════╪════════════════════════════════╪═══════════════════════════════╡
│ 0020010000094 ┆ Không xác định  ┆ ﻿﻿Tã dán Merries size S 82 miế…  ┆ ﻿﻿Tã dán Merries size S 82 miế… │
│ 0020010000098 ┆ Không xác định  ┆ ﻿﻿﻿Bỉm tã quần Merries size M …   ┆ ﻿﻿﻿Bỉm tã quần Merries size M …  │
│ 0024181040235 ┆ Không xác định  ┆ Áo thun bé trai tay ngắn CF    ┆ Áo thun bé trai tay ngắn CF   │
│               ┆                 ┆ B0…                            ┆ B0…                           │
│ 0020010000150 ┆ Khô

In [ ]:
print("Danh sách cột hiện có (trước khi DROP description & description_new):")
print(df_final.columns)
print(f"Tổng số cột: {df_final.width}")


Danh sách cột hiện có (trước khi DROP description & description_new):
['item_id', 'price', 'category_l1', 'category_l2', 'description', 'brand', 'age_group', 'item_type', 'color', 'size', 'description_new', 'gender_target_final', 'description_final']
Tổng số cột: 13


In [ ]:
# ===============================
# 4. DROP description + description_new
# ===============================

df_final = df_final.drop(["description", "description_new"])

print("\nĐã drop description và description_new.")
print(f"Số cột còn lại: {df_final.width}")
print(df_final.columns)


Đã drop description và description_new.
Số cột còn lại: 11
['item_id', 'price', 'category_l1', 'category_l2', 'brand', 'age_group', 'item_type', 'color', 'size', 'gender_target_final', 'description_final']


# Task 4: Chuẩn hóa dữ liệu (nếu có), biến đổi dữ liệu

In [ ]:
print("\n=== THỐNG KÊ BRAND TRƯỚC KHI XỬ LÝ NHÃN HIẾM ===")

brand_stats_before = (
    df_final
    .select(pl.col("brand").value_counts(sort=True))
)

print(brand_stats_before.head(20))
print(f"Tổng số brand khác nhau (before): {brand_stats_before.height}")



=== THỐNG KÊ BRAND TRƯỚC KHI XỬ LÝ NHÃN HIẾM ===
shape: (20, 1)
┌───────────────────────────────┐
│ brand                         │
│ ---                           │
│ struct[2]                     │
╞═══════════════════════════════╡
│ {"Animo",8149}                │
│ {"Không xác định",5472}       │
│ {"CF (ConCung Fashion)",5426} │
│ {"Thương hiệu khác",607}      │
│ {"Con Cưng",484}              │
│ {"TOYCITY",300}               │
│ {"ConCung Good",211}          │
│ {"Nous",140}                  │
│ {"Pigeon",120}                │
│ {"Mesuca",114}                │
│ {"KUKU",100}                  │
│ {"Laluna",75}                 │
│ {"Konbini",63}                │
│ {"Lock&Lock (Hàn Quốc)",59}   │
│ {"CYPRESS TOYS",58}           │
│ {"Đinh Tị",56}                │
│ {"Bobby",56}                  │
│ {"Heinz",54}                  │
│ {"Joie",54}                   │
│ {"NS Minh Long",51}           │
└───────────────────────────────┘
Tổng số brand khác nhau (before): 976


In [ ]:
path_other = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.item_chunk_0-raw -1.parquet"

df_age = pl.read_parquet(path_other).select(["item_id", "age_group_final"])
print(df_age.head())
print("Số dòng trong bảng age:", df_age.height)


shape: (5, 2)
┌───────────────┬──────────────────────────────┐
│ item_id       ┆ age_group_final              │
│ ---           ┆ ---                          │
│ str           ┆ str                          │
╞═══════════════╪══════════════════════════════╡
│ 0502020000004 ┆ Từ 9M                        │
│ 0010290040150 ┆ Từ 36M                       │
│ 0008010000015 ┆ 0-12M                        │
│ 0020010000094 ┆ ["Từ 4M", "3M-6M", "12-36M"] │
│ 0020010000098 ┆ 12-36M                       │
└───────────────┴──────────────────────────────┘
Số dòng trong bảng age: 27323


In [ ]:
df_final = df_final.join(
    df_age,
    on="item_id",
    how="left"
)


In [ ]:
print("Số dòng df_final:", df_final.height)
print("Số dòng unique item_id:", df_final.select(pl.col("item_id").n_unique()).item())


Số dòng df_final: 27323
Số dòng unique item_id: 27323


In [ ]:

total_rows = df_final.height
print("Tổng số dòng:", total_rows)

rows = []

for col_name, dt in zip(df_final.columns, df_final.dtypes):
    # Chỉ tính cho các cột dạng string
    if dt == pl.Utf8:
        cnt_unknown = df_final.filter(pl.col(col_name) == "Không xác định").height
        ratio_unknown = cnt_unknown / total_rows * 100

        rows.append({
            "column": col_name,
            "dtype": str(dt),
            "unknown_count": cnt_unknown,
            "unknown_ratio(%)": round(ratio_unknown, 2),
        })

# Tạo DataFrame kết quả
unknown_stats = pl.DataFrame(rows).sort("unknown_ratio(%)", descending=True)

print("\n=== Thống kê tỷ lệ 'Không xác định' theo cột (string) ===")
print(unknown_stats)

Tổng số dòng: 27323

=== Thống kê tỷ lệ 'Không xác định' theo cột (string) ===
shape: (11, 4)
┌─────────────────────┬────────┬───────────────┬──────────────────┐
│ column              ┆ dtype  ┆ unknown_count ┆ unknown_ratio(%) │
│ ---                 ┆ ---    ┆ ---           ┆ ---              │
│ str                 ┆ str    ┆ i64           ┆ f64              │
╞═════════════════════╪════════╪═══════════════╪══════════════════╡
│ color               ┆ String ┆ 26599         ┆ 97.35            │
│ size                ┆ String ┆ 25099         ┆ 91.86            │
│ age_group           ┆ String ┆ 15803         ┆ 57.84            │
│ description_final   ┆ String ┆ 15436         ┆ 56.49            │
│ gender_target_final ┆ String ┆ 14702         ┆ 53.81            │
│ age_group_final     ┆ String ┆ 10085         ┆ 36.91            │
│ item_type           ┆ String ┆ 9812          ┆ 35.91            │
│ brand               ┆ String ┆ 5472          ┆ 20.03            │
│ item_id             

In [ ]:
df_final = df_final.drop("age_group")

In [ ]:

total_rows = df_final.height
print("Tổng số dòng:", total_rows)

rows = []

for col_name, dt in zip(df_final.columns, df_final.dtypes):
    # Chỉ tính cho các cột dạng string
    if dt == pl.Utf8:
        cnt_unknown = df_final.filter(pl.col(col_name) == "Không xác định").height
        ratio_unknown = cnt_unknown / total_rows * 100

        rows.append({
            "column": col_name,
            "dtype": str(dt),
            "unknown_count": cnt_unknown,
            "unknown_ratio(%)": round(ratio_unknown, 2),
        })

# Tạo DataFrame kết quả
unknown_stats = pl.DataFrame(rows).sort("unknown_ratio(%)", descending=True)

print("\n=== Thống kê tỷ lệ 'Không xác định' theo cột (string) ===")
print(unknown_stats)

Tổng số dòng: 27323

=== Thống kê tỷ lệ 'Không xác định' theo cột (string) ===
shape: (10, 4)
┌─────────────────────┬────────┬───────────────┬──────────────────┐
│ column              ┆ dtype  ┆ unknown_count ┆ unknown_ratio(%) │
│ ---                 ┆ ---    ┆ ---           ┆ ---              │
│ str                 ┆ str    ┆ i64           ┆ f64              │
╞═════════════════════╪════════╪═══════════════╪══════════════════╡
│ color               ┆ String ┆ 26599         ┆ 97.35            │
│ size                ┆ String ┆ 25099         ┆ 91.86            │
│ description_final   ┆ String ┆ 15436         ┆ 56.49            │
│ gender_target_final ┆ String ┆ 14702         ┆ 53.81            │
│ age_group_final     ┆ String ┆ 10085         ┆ 36.91            │
│ item_type           ┆ String ┆ 9812          ┆ 35.91            │
│ brand               ┆ String ┆ 5472          ┆ 20.03            │
│ item_id             ┆ String ┆ 0             ┆ 0.0              │
│ category_l1         

In [ ]:
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.item_chunk_0_done.parquet"

df_final.write_parquet(output_path)

print(f"Đã lưu thành công vào:\n{output_path}")


NameError: name 'df_final' is not defined

# Task 5: Nhóm hãy suy nghĩ xem, với bài toán dự đoán mua hàng, ta có thể tạo mới những đặc trưng nào. Sau đó tiến hành rút trích thêm các đặc trưng. Task này rất quan trọng vì ảnh hưởng hiệu quả của hệ thống.